In [9]:
import pandas as pd
pd.set_option("display.max_columns", None)
import geopandas as gpd
import os

In [2]:
# hexagonos
hex = gpd.read_file(os.path.join("..", "datos", "01_procesos", "malla_hexagonos_cditvallejo.gpkg"))
print(hex.shape)
hex = hex.drop(columns=["left", "top", "right", "bottom", "row_index", "col_index"])
hex = hex[hex["entorno"] != ""]
print(hex.shape)
hex.head(1)

(910, 9)
(441, 3)


,id,entorno,geometry
85,86,Intermunicipal,"POLYGON ((476561.867 2156533.83, 476681.758 21..."


In [3]:
# Manzanas
mzn = gpd.read_file(os.path.join("..", "datos", "01_procesos", "manzanas_datoscenso_zecditvallejo.gpkg"))
mzn.head(1)

,CVEGEO,pob_total,pob_fem,pob_masc,pob_0_14,pob_15_17,pob_18_24,pob_25_29,pob_30_49,pob_50_59,pob_60_mas,pob_alfabeta,pob_analfabeta,pob_sin_escolaridad,pob_basica_comp,pob_posbasica,pob_media_sup,pob_superior,pea,pob_ocupada,pob_ocup_sup,pob_desocupada,pob_no_pea,viv_habitadas,viv_part_hab,ocup_viv,viv_deshabitadas,viv_elect,viv_agua,viv_drenaje,viv_serv_comp,viv_comp,viv_cel,viv_internet,viv_sin_comp_int,viv_1dorm,viv_hacinada,geometry
0,0900200010398005,142.0,71.0,71.0,23.0,3.0,23.0,13.0,33.0,25.0,22.0,119.0,0.0,0.0,21.0,84.0,38.0,34.0,79.0,76.0,28.0,3.0,48.0,42.0,42.0,142.0,14.0,42.0,42.0,42.0,42.0,26.0,39.0,32.0,9.0,15.0,12.0,"MULTIPOLYGON (((483432.725 2153439.42, 483450...."


In [4]:
# Verificar CRS y join espacial por centroide de manzana → hexágono
print(f"CRS hexágonos: {hex.crs}")
print(f"CRS manzanas:  {mzn.crs}")

if hex.crs != mzn.crs:
    mzn = mzn.to_crs(hex.crs)
    print("Manzanas reproyectadas al CRS de hexágonos")

# Centroide de cada manzana para asignarla a un solo hexágono
mzn_c = mzn.copy()
mzn_c["geometry"] = mzn.geometry.centroid

mzn_hex = gpd.sjoin(mzn_c, hex[["id", "entorno", "geometry"]], how="inner", predicate="within")

print(f"\nManzanas totales:                {mzn.shape[0]:,}")
print(f"Manzanas con hexágono asignado:  {mzn_hex.shape[0]:,}")
print(f"Manzanas sin cobertura:          {mzn.shape[0] - mzn_hex.shape[0]:,}")
mzn_hex[["CVEGEO", "id", "entorno"]].head(3)

CRS hexágonos: EPSG:32614
CRS manzanas:  EPSG:32614

Manzanas totales:                5,700
Manzanas con hexágono asignado:  5,548
Manzanas sin cobertura:          152


,CVEGEO,id,entorno
0,0900200010398005,588,Intermunicipal
1,0900200010398004,588,Intermunicipal
2,0900200010398001,613,Intermunicipal


In [5]:
# Columnas de conteo crudo a agregar por hexágono
cols_conteo = [
    "pob_total", "pob_fem", "pob_masc", "pob_0_14", "pob_15_17",
    "pob_18_24", "pob_25_29", "pob_30_49", "pob_50_59", "pob_60_mas",
    "pob_alfabeta", "pob_analfabeta", "pob_sin_escolaridad", "pob_basica_comp",
    "pob_posbasica", "pob_media_sup", "pob_superior", "pea", "pob_ocupada",
    "pob_ocup_sup", "pob_desocupada", "pob_no_pea", "viv_habitadas",
    "viv_part_hab", "ocup_viv", "viv_deshabitadas", "viv_elect", "viv_agua",
    "viv_drenaje", "viv_serv_comp", "viv_comp", "viv_cel", "viv_internet",
    "viv_sin_comp_int", "viv_1dorm", "viv_hacinada"
]

# Suma por hexágono (NaN de datos confidenciales se omiten al sumar)
agg = mzn_hex.groupby("id")[cols_conteo].sum().reset_index()
n_mzn = mzn_hex.groupby("id").size().reset_index(name="n_manzanas")
agg = agg.merge(n_mzn, on="id")

print(f"Hexágonos con datos agregados: {agg.shape[0]}")
print(f"Conteos manzanas por hexágono — media: {agg['n_manzanas'].mean():.1f}, min: {agg['n_manzanas'].min()}, max: {agg['n_manzanas'].max()}")
agg[["id", "n_manzanas", "pob_total", "viv_habitadas"]].head(3)

Hexágonos con datos agregados: 415
Conteos manzanas por hexágono — media: 13.4, min: 1, max: 56


,id,n_manzanas,pob_total,viv_habitadas
0,86,4,556.0,188.0
1,88,2,79.0,29.0
2,89,10,1644.0,469.0


In [6]:
def agregar_porcentajes(gdf):
    """
    Agrega columnas pct_* al GeoDataFrame de hexágonos con datos censales agregados.
    Misma lógica y denominadores que nb_informacionCensoAnalisis.ipynb.
    """
    df = gdf.copy()

    def pct(num, den):
        return (num / den.replace(0, float('nan')) * 100).round(2)

    # Grupos de edad (denominador: pob_total)
    for col in ['pob_fem', 'pob_masc', 'pob_0_14', 'pob_15_17',
                'pob_18_24', 'pob_25_29', 'pob_30_49', 'pob_50_59', 'pob_60_mas']:
        df[f'pct_{col.removeprefix("pob_")}'] = pct(df[col], df['pob_total'])

    # Educación (denominador: pob 15+ años)
    den_15_mas = df['pob_total'] - df['pob_0_14']
    for col in ['pob_alfabeta', 'pob_analfabeta', 'pob_sin_escolaridad',
                'pob_basica_comp', 'pob_posbasica', 'pob_media_sup', 'pob_superior']:
        df[f'pct_{col.removeprefix("pob_")}'] = pct(df[col], den_15_mas)

    # Actividad económica
    df['pct_pea']        = pct(df['pea'],            den_15_mas)
    df['pct_no_pea']     = pct(df['pob_no_pea'],     den_15_mas)
    df['pct_ocupada']    = pct(df['pob_ocupada'],     df['pea'])
    df['pct_desocupada'] = pct(df['pob_desocupada'],  df['pea'])
    df['pct_ocup_sup']   = pct(df['pob_ocup_sup'],    df['pob_ocupada'])

    # Vivienda (denominador: viv_part_hab)
    for col in ['viv_elect', 'viv_agua', 'viv_drenaje', 'viv_serv_comp',
                'viv_comp', 'viv_cel', 'viv_internet', 'viv_sin_comp_int',
                'viv_1dorm', 'viv_hacinada']:
        df[f'pct_{col}'] = pct(df[col], df['viv_part_hab'])

    # Desocupación de viviendas (base: total viviendas = hab + deshabitadas)
    df['pct_viv_deshabitadas'] = pct(
        df['viv_deshabitadas'],
        df['viv_habitadas'] + df['viv_deshabitadas']
    )

    return df


# Aplicar a los conteos agregados por hexágono
agg_pct = agregar_porcentajes(agg)

# Unir con geometría de hexágonos (left join para conservar todos los hexágonos)
hex_datos = hex.merge(agg_pct, on="id", how="left")

# Densidad poblacional usando el área real del hexágono
hex_datos["area_m2"]    = hex_datos.geometry.area
hex_datos["den_pob_ha"] = (hex_datos["pob_total"] / (hex_datos["area_m2"] / 10_000)).round(2)

cols_pct = [c for c in hex_datos.columns if c.startswith("pct_")]
print(f"Hexágonos totales:     {hex_datos.shape[0]}")
print(f"Con datos censales:    {hex_datos['pob_total'].notna().sum()}")
print(f"Sin datos (sin manzanas): {hex_datos['pob_total'].isna().sum()}")
print(f"Columnas pct_ generadas: {len(cols_pct)}")
hex_datos[["id", "entorno", "n_manzanas", "pob_total", "den_pob_ha"] + cols_pct[:4]].head(3)

Hexágonos totales:     441
Con datos censales:    415
Sin datos (sin manzanas): 26
Columnas pct_ generadas: 32


,id,entorno,n_manzanas,pob_total,den_pob_ha,pct_fem,pct_masc,pct_0_14,pct_15_17
0,86,Intermunicipal,4.0,556.0,39.92,54.50,45.50,10.43,3.78
1,87,Intermunicipal,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,88,Intermunicipal,2.0,79.0,5.67,54.43,45.57,5.06,0.00


In [7]:
hex_datos.head(1)

,id,entorno,geometry,pob_total,pob_fem,pob_masc,pob_0_14,pob_15_17,pob_18_24,pob_25_29,pob_30_49,pob_50_59,pob_60_mas,pob_alfabeta,pob_analfabeta,pob_sin_escolaridad,pob_basica_comp,pob_posbasica,pob_media_sup,pob_superior,pea,pob_ocupada,pob_ocup_sup,pob_desocupada,pob_no_pea,viv_habitadas,viv_part_hab,ocup_viv,viv_deshabitadas,viv_elect,viv_agua,viv_drenaje,viv_serv_comp,viv_comp,viv_cel,viv_internet,viv_sin_comp_int,viv_1dorm,viv_hacinada,n_manzanas,pct_fem,pct_masc,pct_0_14,pct_15_17,pct_18_24,pct_25_29,pct_30_49,pct_50_59,pct_60_mas,pct_alfabeta,pct_analfabeta,pct_sin_escolaridad,pct_basica_comp,pct_posbasica,pct_media_sup,pct_superior,pct_pea,pct_no_pea,pct_ocupada,pct_desocupada,pct_ocup_sup,pct_viv_elect,pct_viv_agua,pct_viv_drenaje,pct_viv_serv_comp,pct_viv_comp,pct_viv_cel,pct_viv_internet,pct_viv_sin_comp_int,pct_viv_1dorm,pct_viv_hacinada,pct_viv_deshabitadas,area_m2,den_pob_ha
0,86,Intermunicipal,"POLYGON ((476561.867 2156533.83, 476681.758 21...",556.0,303.0,253.0,58.0,21.0,49.0,29.0,145.0,91.0,163.0,493.0,0.0,4.0,26.0,431.0,93.0,293.0,283.0,282.0,225.0,0.0,230.0,188.0,181.0,556.0,8.0,188.0,188.0,188.0,188.0,157.0,180.0,169.0,12.0,27.0,0.0,4.0,54.5,45.5,10.43,3.78,8.81,5.22,26.08,16.37,29.32,99.0,0.0,0.8,5.22,86.55,18.67,58.84,56.83,46.18,99.65,0.0,79.79,103.87,103.87,103.87,103.87,86.74,99.45,93.37,6.63,14.92,0.0,4.08,139291.295221,39.92


In [8]:
# Exportar
ruta_procesos = os.path.join("..", "datos", "01_procesos")
ruta_finales  = os.path.join("..", "datos", "02_finales")
ruta_docs     = os.path.join("..", "docs", "datos")

os.makedirs(ruta_finales, exist_ok=True)
os.makedirs(ruta_docs, exist_ok=True)

# GeoPackage en procesos (CRS proyectado, para uso en SIG)
hex_datos.to_file(
    os.path.join(ruta_procesos, "hexagonos_censo_pct_zecditvallejo.gpkg"),
    driver="GPKG"
)

# GeoJSON en WGS84 para geovisor
hex_datos.to_crs("EPSG:4326").to_file(
    os.path.join(ruta_finales, "hexagonos_censo_pct_zecditvallejo.geojson")
)
hex_datos.to_crs("EPSG:4326").to_file(
    os.path.join(ruta_docs, "hexagonos_censo_pct_zecditvallejo.geojson")
)

print("Exportados:")
print(f"  hexagonos_censo_pct_zecditvallejo.gpkg     → datos/01_procesos/")
print(f"  hexagonos_censo_pct_zecditvallejo.geojson  → datos/02_finales/ y docs/datos/")
print(f"  {hex_datos.shape[0]} hexágonos | {hex_datos.shape[1]} columnas")

Exportados:
  hexagonos_censo_pct_zecditvallejo.gpkg     → datos/01_procesos/
  hexagonos_censo_pct_zecditvallejo.geojson  → datos/02_finales/ y docs/datos/
  441 hexágonos | 74 columnas


In [10]:
# Disolver hexágonos por entorno → polígonos para overlay de contornos en el geovisor
entorno_contornos = (
    hex_datos[['entorno', 'geometry']]
    .dissolve(by='entorno')
    .reset_index()
)

print(f"Entornos disueltos: {entorno_contornos.shape[0]}")
print(entorno_contornos['entorno'].tolist())

entorno_contornos.to_crs("EPSG:4326").to_file(
    os.path.join(ruta_finales, "entornos_contornos_zecditvallejo.geojson")
)
entorno_contornos.to_crs("EPSG:4326").to_file(
    os.path.join(ruta_docs, "entornos_contornos_zecditvallejo.geojson")
)
print("Exportado: entornos_contornos_zecditvallejo.geojson")

Entornos disueltos: 3
['Barrio', 'Intermunicipal', 'local']
Exportado: entornos_contornos_zecditvallejo.geojson
